# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the [FAIR² dataset](https://doi.org/10.71728/senscience.y7m0-f273) using the `mlcroissant` library. We will use Croissant schema `@id` references throughout for reproducibility.

### Dataset Source
The dataset source is provided via [this Croissant schema URL](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

In [ ]:
# Ensure `mlcroissant` is installed (run once)
!pip install -U mlcroissant

## 1. Data Loading

Load dataset metadata and inspect a summary using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the Croissant dataset via URL
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object (not via dict subscripting)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Temporal coverage: {metadata.temporal_coverage}")
print(f"License: {metadata.license}")

## 2. Data Overview

List available record sets, fields, and their `@id` properties. All references use full `@id` paths for clarity.


In [ ]:
# List available record sets and fields (by their `@id`)
record_sets = [record_set for record_set in dataset.record_sets]
print(f"Number of record sets found: {len(record_sets)}\n")

for rs in record_sets:
    print(f"\nRecord set: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {getattr(rs, 'description', None)}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.id}  (name: {field.name}, type: {field.data_type})")

## 3. Data Extraction

Load records from each record set into a pandas DataFrame for analysis. Use only `@id` for referencing record sets and fields.

In [ ]:
# Collect all record set ids
record_set_ids = [rs.id for rs in record_sets]

# Load records for each record set into DataFrames (referenced by their `@id`)
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set {record_set_id} with shape {df.shape}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Preview the columns of the first available record set
if record_set_ids:
    first_rs = record_set_ids[0]
    if first_rs in dataframes:
        print(f"\nColumns for record set {first_rs}:")
        print(dataframes[first_rs].columns.tolist())
        display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps such as filtering, normalizing, and grouping. All references to fields and columns are made via their `@id`. We'll automatically select a numeric field if available and use its `@id` for demonstration.

In [ ]:
# We'll use the first DataFrame with at least one numeric field.
import numpy as np

target_record_set_id = None
numeric_field_id = None
group_field_id = None

for rs in record_sets:
    rs_id = rs.id
    df = dataframes.get(rs_id)
    if df is not None and not df.empty:
        # Identify a numeric field by Croissant data type
        for field in rs.fields:
            # Schema: 'Float', 'Integer', 'Number'
            if str(field.data_type).lower() in ['float', 'integer', 'number']:
                numeric_field_id = field.id
                target_record_set_id = rs_id
                # Use the next available non-numeric field for grouping
                for group_field in rs.fields:
                    if str(group_field.data_type).lower() not in ['float', 'integer', 'number']:
                        group_field_id = group_field.id
                        break
                break
    if target_record_set_id is not None:
        break

if target_record_set_id and numeric_field_id:
    target_df = dataframes[target_record_set_id]

    # Check if the numeric field exists in DataFrame columns
    # (Croissant @id may differ slightly from column naming; best effort match by name or id)
    field_column = numeric_field_id if numeric_field_id in target_df.columns else None
    if not field_column:
        # Try using the field short name
        field_column = rs.get_field(numeric_field_id).name if hasattr(rs, 'get_field') and hasattr(rs.get_field(numeric_field_id), 'name') else None
        if field_column and field_column not in target_df.columns:
            field_column = None
    if not field_column and target_df.select_dtypes(include=[np.number]).shape[1] > 0:
        # As fallback, just take the first numeric column
        field_column = target_df.select_dtypes(include=[np.number]).columns[0]
        numeric_field_id = field_column

    print(f"Selected numeric field: {numeric_field_id}")

    threshold = 10
    if field_column:
        # Filtering
        filtered_df = target_df[target_df[field_column] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold} (count: {len(filtered_df)}):")
        display(filtered_df.head())

        # Normalization
        norm_col = f"{field_column}_normalized"
        filtered_df[norm_col] = (filtered_df[field_column] - filtered_df[field_column].mean()) / filtered_df[field_column].std(ddof=0)

        print(f"\nNormalized '{field_column}' for filtered records:")
        display(filtered_df[[field_column, norm_col]].head())

        # Group by another field if available
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[field_column].mean().reset_index()
            print(f"\nGroup mean of {field_column} by '{group_field_id}':")
            display(grouped_df.head())
        else:
            print("No suitable group-by field available for this record set.")
    else:
        print("Could not find a suitable numeric field in the DataFrame columns.")
else:
    print("Could not find a suitable record set with numeric field for EDA.")

## 5. Visualization

Visualize the distribution of the selected numeric field and its normalized counterpart. (This visualization will run only if the required field and data are available.)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization will depend on prior code success
if 'filtered_df' in locals() and len(filtered_df) > 0 and field_column:
    plt.figure(figsize=(10,4))
    sns.histplot(filtered_df[field_column], kde=True, bins=30, label=field_column)
    plt.title(f"Distribution of {numeric_field_id} in filtered records")
    plt.xlabel(field_column)
    plt.legend()
    plt.show()
    
    if norm_col in filtered_df:
        plt.figure(figsize=(10,4))
        sns.histplot(filtered_df[norm_col], kde=True, bins=30, color='salmon', label=norm_col)
        plt.title(f"Distribution of normalized {numeric_field_id}")
        plt.xlabel(norm_col)
        plt.legend()
        plt.show()
else:
    print("Visualization not available due to missing or insufficient data.")

## 6. Conclusion

- We used the `mlcroissant` library to load, inspect, and analyze the FAIR² dataset provided in Croissant format.
- All exploration referenced Croissant `@id`s for reproducibility and transparency.
- We demonstrated loading record sets, extracting fields, basic filtering, normalization, and simple visualizations.
- This approach is extensible to more complex data processing tasks and different Croissant-compliant datasets.

#### Next steps
- Explore additional fields, relationships, and metadata exports.
- Use the record set and field `@id`s for reproducibility in pipelines and documentation.
- For more comprehensive analyses, consult the [FAIR² documentation](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) for context and schema details.